# Agents

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
	index=index,
	llm_client=openai_client,
	instructions=instructions,
)

In [4]:
search_tool = {
	"type": "function",
	"function": {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [5]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
	)

In [6]:
messages = [
  {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [7]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	user="llm-zoomcamp",
	stream=False,
	tools=[search_tool]
)

In [8]:
import json

if response.choices[0].finish_reason == "tool_calls":
	message = response.choices[0].message
	function_call = response.choices[0].message.tool_calls[0].function
  
	if function_call.name == "search":
		# retrieve tool call params and call search function
		args = json.loads(function_call.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)
		
		# add the model response and tool call results to the message history
		messages.append(message)
		messages.append({
			"role": "tool",
			"tool_call_id": response.choices[0].message.tool_calls[0].id,
			"content": result_json
		})

		# send new prompt with updated message history
		response = openai_client.chat.completions.create(
			model="gpt-4o-mini",
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		print(response.choices[0].message.content)
	else:
		print("Unable to perform search")
else:
	print(response.choices[0].message.content)

Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You don't need to register formally; you're accepted and can start learning and submitting homework as long as the submission form is open.


## Agentic loop

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [10]:
def make_call(call):
	if call.function.name == "search":
		args = json.loads(call.function.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)

		return {
			"role": "tool",
			"tool_call_id": call.id,
			"content": result_json
		}

In [11]:
question = "I just discovered the course. Can I join it?"

In [12]:
def agent_loop(instructions, question, model="gpt-4o-mini") -> str:
	messages = [
		{"role": "developer", "content": instructions},
		{"role": "user", "content": question},
	]

	last_answer = "Could not generate a response"
	it = 1

	while True:
		print(f"iteration #{it}...")	
		has_function_calls = False
		
		response = openai_client.chat.completions.create(
			model=model,
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		message = response.choices[0].message
		messages.append(message)
		
		for item in response.choices:
			if item.finish_reason == "tool_calls":
				tool_call = message.tool_calls[0]
				print("function_call:", tool_call.function.name, tool_call.function.arguments)
				call_output = make_call(tool_call)
				messages.append(call_output)
				has_function_calls = True
			elif item.finish_reason == "stop":
				last_answer = item.message.content
				print(item.message.content)
		
		it += 1
		if has_function_calls == False:
			break

	return last_answer

In [13]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


function_call: search {"query":"run Olama locally"}
iteration #2...
Yes, you can run the course locally instead of using Codespaces. The passage suggests that while Codespaces is designed for convenience, a local setup is a viable option if you are comfortable configuring the necessary tools like Python, `uv`, Jupyter, Docker, and others required for the module.

If you choose to go this route, make sure to document your setup and ensure that your environment is reproducible. 

If you have any specific questions about the tools or setup required for running Olama locally, feel free to ask! Are there any other areas you would like to explore?


'Yes, you can run the course locally instead of using Codespaces. The passage suggests that while Codespaces is designed for convenience, a local setup is a viable option if you are comfortable configuring the necessary tools like Python, `uv`, Jupyter, Docker, and others required for the module.\n\nIf you choose to go this route, make sure to document your setup and ensure that your environment is reproducible. \n\nIf you have any specific questions about the tools or setup required for running Olama locally, feel free to ask! Are there any other areas you would like to explore?'

## Agent frameworks

In [14]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [15]:
def search(query: str) -> dict[str, str]:
	"""
	Search the FAQ database for entries matching the given query.
	"""
	return index.search(
		query,
		num_results=5,
		boost_dict={"question": 3.0, "section": 0.5},
		filter_dict={"course": "llm-zoomcamp"}
	)

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'function': {'name': 'search',
   'description': 'Search the FAQ database for entries matching the given query.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'Search query text to look up in the course FAQ.'}},
    'required': ['query'],
    'additionalProperties': False}}}]

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [23]:
runner = OpenAIResponsesRunner(
	tools=agent_tools,
	developer_prompt=instructions,
	chat_interface=chat_interface,
	llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
	prompt="How do I run Olama locally?",
	callback=callback,
)